## Data merging and cleaning

### Cleaning

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [1]:
import pandas as pd
all_models = pd.read_excel('all_models.xlsx')

In [2]:
all_models.columns

Index(['question', 'GPT-4o Output', 'Allam Output', 'LLaMA Output'], dtype='object')

In [ ]:
all_models.drop(columns=["Unnamed: 0", "Unnamed: 0.1"], inplace=True)

In [3]:
all_models.rename(columns={"LLaMA Output":"LLaMA", "Allam Output": "ALLaM", "GPT-4o Output":"GPT-4o"}, inplace=True)

In [4]:
all_models.columns

Index(['question', 'GPT-4o', 'ALLaM', 'LLaMA'], dtype='object')

In [5]:
import json
import re
from ast import literal_eval
import pandas as pd

# List of all expected entity keys
expected_keys = ["Symptom", "Diagnosis", "Treatment", "Drug", "Dosage", "Duration", "Strength", "Unit", "Risk"]

def clean_model_output(text):
    """
    Cleans and normalizes model output (like Allam or LLaMA) into a JSON list
    with expected medical keys.
    """
    try:
        if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
            return json.dumps([{k: "" for k in expected_keys}], ensure_ascii=False)

        # Remove prefix text before the first "[" (e.g., "الإخراج:")
        text = text.strip()
        text = re.sub(r'^.*?\[', '[', text, flags=re.DOTALL)

        # Extract JSON-like list string
        match = re.search(r'\[.*\]', text, re.DOTALL)
        if not match:
            return json.dumps([{k: "" for k in expected_keys}], ensure_ascii=False)

        json_str = match.group(0)

        # Try to parse using ast.literal_eval first, fallback to json.loads
        try:
            parsed = literal_eval(json_str)
        except Exception:
            parsed = json.loads(json_str)

        # Normalize each entry to include all keys
        if isinstance(parsed, list):
            cleaned = []
            for entry in parsed:
                if isinstance(entry, dict):
                    cleaned_dict = {k: str(entry.get(k, "")).strip() for k in expected_keys}
                    cleaned.append(cleaned_dict)
            return json.dumps(cleaned, ensure_ascii=False)
        else:
            return json.dumps([{k: "" for k in expected_keys}], ensure_ascii=False)

    except Exception:
        return json.dumps([{k: "" for k in expected_keys}], ensure_ascii=False)

In [6]:
df = all_models.copy()

In [7]:
import re

def clean_allam_output(text: str) -> str:
    # Normalize
    s = str(text).strip()

    # Remove Arabic/English "Output:" prefixes if present
    s = re.sub(r"^(?:الإخراج|Output)\s*:\s*", "", s, flags=re.IGNORECASE)

    if not s:
        return "no answer"

    # 1) If there is any complete JSON-like list, keep only the FIRST one
    # (and drop anything after its proper closing)
    m = re.search(r"\[\s*\{.*?\}\s*\]", s, flags=re.DOTALL)
    if m:
        return m.group(0)

    # 2) Otherwise, look for a trailing, *started* list and auto-close it.
    #    We take the LAST '[{' to avoid earlier false starts inside the text.
    start_idx = s.rfind("[{")
    if start_idx != -1:
        candidate = s[start_idx:].strip()

        # If we can find a proper closing inside the tail, cut at the first one.
        close_match = re.search(r"\}\s*\]", candidate, flags=re.DOTALL)
        if close_match:
            end = close_match.end()
            return candidate[:end]

        # Not properly closed -> auto-close
        return candidate + "}]"

    # 3) No list at all → no answer
    return "no answer"

In [8]:
df["ALLaM Cleaned"] = df["ALLaM"].apply(clean_allam_output)

In [9]:
from google.colab import files
df.to_excel("all_cleaned.xlsx", index=False, engine='openpyxl')
files.download("all_cleaned.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
df["LLaMA Cleaned"] = df["LLaMA"].apply(clean_model_output)
df["GPT-4o Cleaned"] = df["GPT-4o"].apply(clean_model_output)

In [11]:
df.to_excel("all_cleaned.xlsx", index=False, engine='openpyxl')

In [12]:
from google.colab import files
files.download("all_cleaned.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
df.head()

,question,GPT-4o,ALLaM,LLaMA,ALLaM Cleaned,LLaMA Cleaned,GPT-4o Cleaned
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,"[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","الإخراج:\n[{""Symptom"": ""الخوف من عدم الإنجاب م...","[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","[{""Symptom"": ""الخوف من عدم الإنجاب مستقبلاً"", ...","[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,"[{""Symptom"": ""أذي نفسي"", ""Diagnosis"": """", ""Tre...","الإخراج:\n[{""Symptom"": ""أذي نفسي"", ""Diagnosis""...","[{""Symptom"": ""أذي نفسي"", ""Diagnosis"": """", ""Tre...","[{""Symptom"": ""أذي نفسي"", ""Diagnosis"": """", ""Tre...","[{""Symptom"": ""أذي نفسي"", ""Diagnosis"": """", ""Tre...","[{""Symptom"": ""أذي نفسي"", ""Diagnosis"": """", ""Tre..."
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,"[{""Symptom"": ""تكرار كلمة معينة، الاحباط"", ""Dia...","الإخراج:\n[{""Symptom"": ""تكرار كلمة معينة لمدة ...","[{""Symptom"": ""تكرار كلمة معينة، بلاحباط"", ""Dia...","[{""Symptom"": ""تكرار كلمة معينة لمدة طويلة، الا...","[{""Symptom"": ""تكرار كلمة معينة، بلاحباط"", ""Dia...","[{""Symptom"": ""تكرار كلمة معينة، الاحباط"", ""Dia..."
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,"[{""Symptom"": ""انعزال، انطوائية، تعامل بسلبية"",...","الإخراج:\n[{""Symptom"": ""اكتئاب، فوبيا من المجت...","[{""Symptom"": ""انعزال، وانطوائية، وتعامل بسلبية...","[{""Symptom"": ""اكتئاب، فوبيا من المجتمع, انعزال...","[{""Symptom"": ""انعزال، وانطوائية، وتعامل بسلبية...","[{""Symptom"": ""انعزال، انطوائية، تعامل بسلبية"",..."
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,"[{""Symptom"": ""قلق شديد مزمن، خوف كبير"", ""Diagn...","الإخراج:\n[{""Symptom"": ""قلق شديد، خوف، الوسواس...","[{""Symptom"": ""قلق شديد مزمن، خوف كبير، الوسواس...","[{""Symptom"": ""قلق شديد، خوف، الوسواس القهري"", ...","[{""Symptom"": ""قلق شديد مزمن، خوف كبير، الوسواس...","[{""Symptom"": ""قلق شديد مزمن، خوف كبير"", ""Diagn..."


##  Import Libraries & Load the data

In [14]:
!pip install openai tqdm

In [15]:
import pandas as pd
import openai
import json
from tqdm import tqdm
import os

In [16]:
len(df)

499

In [17]:
df.tail()

,question,GPT-4o,ALLaM,LLaMA,ALLaM Cleaned,LLaMA Cleaned,GPT-4o Cleaned
494,قلق وتوتر ووسواس قهري وتأتأة في الكلام وعجز دا...,"[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","الإخراج:\n[{""Symptom"": ""قلق، توتر، وسواس قهري،...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ..."
495,اعاني من الخوف وتنبؤات من كلام اهلي وبصمتهم ان...,"[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm...","الإخراج:\n[{""Symptom"": ""الرهاب""، ""Diagnosis"": ...","[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm...","[{""Symptom"": ""الرهاب""، ""Diagnosis"": ""اضطراب ال...","[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm...","[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm..."
496,لدي خمس سنوات علاج من الذهانية ولم اشفى تماما ...,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","الإخراج:\n[{""Symptom"": ""الذهانية"", ""Diagnosis""...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","[{""Symptom"": ""الذهانية"", ""Diagnosis"": ""الذهان""...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre..."
497,أحس بفقدان التركيز كأنني أنسى معلومة أخبرتني ب...,"[{""Symptom"": ""فقدان التركيز، أحس بالنوم وقت ال...","الإخراج:\n[{""Symptom"": ""فقدان التركيز، النوم، ...","[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":..."
498,انا احس فيني الاكتئاب الجزئي او عسر المزاج نفس...,"[{""Symptom"": ""ماعندي شهيه، ماعندي شغف، مالي نف...","الإخراج:\n[{""Symptom"": ""الإكتئاب الجزئي، اعراض...","[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، ماع...","[{""Symptom"": ""الإكتئاب الجزئي، اعراض اكتئاب جز...","[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، ماع...","[{""Symptom"": ""ماعندي شهيه، ماعندي شغف، مالي نف..."


In [18]:
df.columns

Index(['question', 'GPT-4o', 'ALLaM', 'LLaMA', 'ALLaM Cleaned',
       'LLaMA Cleaned', 'GPT-4o Cleaned'],
      dtype='object')

## Define the Judge Prompt

In [19]:
def build_prompt(question_text, ner_output):
    return f"""
    You are an expert medical evaluator. Your task is to act as an impartial judge and evaluate the quality of named entity extraction performed by a single AI model. We will provide you with a medical question,
    a model-generated list of extracted entities, and a set of evaluation criteria. You should first read the question carefully, then evaluate the extracted entities based on the following two criteria: Correctness and Completeness.

    For each criterion:
    - Use only the question text as the source of truth.
    - Refer strictly to the entity definitions provided.
    - Do not infer or hallucinate missing information.
    - Output your final verdict using one of the valid options for each criterion (see below).
    - Provide a short explanation for your decision.

    ## Question:
    {question_text}

    ## Model Output (NER Prediction):
    {ner_output}

    ## Entity Schema:
    Evaluate only the following entity types and definitions:
    - Symptom: Observable psychological or behavioral symptoms (e.g., anxiety, insomnia).
    - Diagnosis: Mentioned or diagnosed mental health disorders (e.g., generalized anxiety disorder).
    - Treatment: Non-pharmacological therapy methods (e.g., cognitive behavioral therapy).
    - Drug: Names of medications or pharmaceutical substances (e.g., Xanax, Sertraline).
    - Dosage: Dosage instructions (e.g., twice a day, 50 mg).
    - Duration: Duration of medication use or symptom persistence (e.g., for one week, for 3 months).
    - Strength: Concentration of the medication (e.g., 500, 1.0).
    - Unit: Unit of measurement for strength (e.g., mg, ml, g).
    - Risk: Indicators of psychological or physical risk (e.g., suicidal thoughts, self-harm).

    ## Evaluation Criteria

    [The Start of Correctness Criterion]
    Does the model correctly extract the entity spans and assign the correct entity labels **based only on the question text**?

    - accurate: Most or all entities are correct, with minor or no issues.
    - mixed: Some correct extractions alongside boundary or label errors.
    - poor: Many incorrect, misaligned, mislabeled, or spurious entities.
    [The End of Correctness Criterion]

    [The Start of Completeness Criterion]
    How well did the model cover all important entities mentioned **in the question**?

    - full: All relevant entities present in the question are extracted.
    - partial: Some entities are extracted (about half or more).
    - low: Less than half of the important entities are captured.
    [The End of Completeness Criterion]

    ## Output Format
    Please return your evaluation in the following exact format as a markdown-style JSON block:

    {{
      "correctness": {{
        "verdict": "accurate/mixed/inaccurate",
        "reason": "Your detailed reasoning here..."
      }},
      "completeness": {{
        "verdict": "full/partial/missing",
        "reason": "Your detailed reasoning here..."
      }},
      "notes": "Optional comments if needed"
    }}

    """


## DeepSeek

In [24]:
os.environ["DEEPSEEK_API_KEY"] = "Put your key here"
import re
import json
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

def score_with_deepseek(prompt):
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1000,
            temperature=0
        )
        content = response.choices[0].message.content.strip()

        # Try to extract JSON block
        match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", content, re.DOTALL)
        if not match:
            match = re.search(r"(\{.*\})", content, re.DOTALL) # سويت تعديل للتجربة
        if match:
           # raw = re.sub(r'(?<!\\)\n', r'\\n', match.group(1)) #new addition for my problem
            return json.loads(match.group(1))
        else:
            raise ValueError("No JSON block found")

    except Exception as e:
        return {
            "correctness": {"verdict": "error", "reason": str(e)},
            "completeness": {"verdict": "error", "reason": str(e)},
            "notes": ""
        }

In [25]:
test = score_with_deepseek(build_prompt(df.iloc[0]["question"], df.iloc[0]["GPT-4o Cleaned"]))
print(test)

{'correctness': {'verdict': 'accurate', 'reason': "The model correctly identified 'الخوف من عدم الإنجاب' (fear of not having children) as a Symptom, which aligns with the definition of observable psychological symptoms. No other entities were present in the question, and the model appropriately left other fields empty."}, 'completeness': {'verdict': 'full', 'reason': 'The only relevant entity in the question is the fear of not having children, which was extracted. No other symptoms, diagnoses, treatments, drugs, dosages, durations, strengths, units, or risks are mentioned in the question.'}, 'notes': 'The extraction is accurate and complete given the limited entities in the question.'}


In [26]:
# Iterate and collect results
model_cols = ["LLaMA Cleaned", "ALLaM Cleaned", "GPT-4o Cleaned"]

In [ ]:
from collections import Counter

# Reset before each new run
summary = {col: {"correctness": Counter(), "completeness": Counter()} for col in model_cols}
results = {col: [] for col in model_cols}

In [ ]:
from tqdm import tqdm

for col in model_cols:
    for i, row in tqdm(df.iterrows(), total=len(df), desc=f"Scoring {col}"):
        prompt = build_prompt(row['question'], row[col])
        score = score_with_deepseek(prompt)

        results[col].append(score)

        # ✅ Update verdict counts
        if isinstance(score, dict):
            c = score.get("correctness", {}).get("verdict", "error")
            cm = score.get("completeness", {}).get("verdict", "error")
            summary[col]["correctness"][c] += 1
            summary[col]["completeness"][cm] += 1

In [27]:
results_dict = {}

## LLaMA

In [41]:
from tqdm import tqdm
import json

model_col = "LLaMA Cleaned"
judgment_field = "LLaMA Judgment"


for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating {model_col}"):
    key = row["question"]

    # Initialize if new
    if key not in results_dict:
        results_dict[key] = {
            model_col: row[model_col]
        }

    # Skip if already evaluated
    if judgment_field in results_dict[key]:
        continue

    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)
    results_dict[key][judgment_field] = score

# Save after full column is done
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)


Evaluating LLaMA Cleaned: 100%|██████████| 499/499 [00:00<00:00, 19979.93it/s]


In [29]:
llama = pd.DataFrame(results_dict.values())

In [30]:
llama.tail()

,LLaMA Cleaned,LLaMA Judgment
494,"[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","{'correctness': {'verdict': 'mixed', 'reason':..."
495,"[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm...","{'correctness': {'verdict': 'mixed', 'reason':..."
496,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","{'correctness': {'verdict': 'mixed', 'reason':..."
497,"[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","{'correctness': {'verdict': 'mixed', 'reason':..."
498,"[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، ماع...","{'correctness': {'verdict': 'mixed', 'reason':..."


In [31]:
llama.insert(0, 'Question', df['question'])

In [32]:
llama.head()

,Question,LLaMA Cleaned,LLaMA Judgment
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,"[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","{'correctness': {'verdict': 'accurate', 'reaso..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,"[{""Symptom"": ""أذي نفسي"", ""Diagnosis"": """", ""Tre...","{'correctness': {'verdict': 'mixed', 'reason':..."
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,"[{""Symptom"": ""تكرار كلمة معينة، بلاحباط"", ""Dia...","{'correctness': {'verdict': 'mixed', 'reason':..."
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,"[{""Symptom"": ""انعزال، وانطوائية، وتعامل بسلبية...","{'correctness': {'verdict': 'accurate', 'reaso..."
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,"[{""Symptom"": ""قلق شديد مزمن، خوف كبير، الوسواس...","{'correctness': {'verdict': 'mixed', 'reason':..."


In [33]:
llama.to_excel("llama_judgments.xlsx", index=False)

from google.colab import files
files.download("llama_judgments.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
llama.to_csv("llama_judgments.csv", index=False)
files.download("llama_judgments.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
import pandas as pd

flattened = []

# Loop through each question entry in results_dict
for question, data in results_dict.items():
    for model_col in ["LLaMA Cleaned", "ALLaM Cleaned", "GPT-4o Cleaned"]:
        # Find the corresponding judgment field for that model
        judgment_field = model_col.replace("Cleaned", "Judgment").strip()

        if judgment_field in data:
            score = data[judgment_field]
            if isinstance(score, dict):
                flattened.append({
                    "Question": question,
                    "Model": model_col,
                    "Correctness": score["correctness"]["verdict"],
                    "Correctness Reason": score["correctness"]["reason"],
                    "Completeness": score["completeness"]["verdict"],
                    "Completeness Reason": score["completeness"]["reason"],
                    "Notes": score.get("notes", "")
                })

# Create a DataFrame and export
df_results = pd.DataFrame(flattened)
df_results.to_excel("NER_Evaluation_by_DeepSeek.xlsx", index=False)
print(f"Exported {len(df_results)} rows to NER_Evaluation_by_DeepSeek.xlsx")

Exported 499 rows to NER_Evaluation_by_DeepSeek.xlsx


In [36]:
df_results.head()

,Question,Model,Correctness,Correctness Reason,Completeness,Completeness Reason,Notes
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,LLaMA Cleaned,accurate,The model correctly identified 'الخوف من عدم ا...,full,The only relevant entity in the question is th...,The extraction is accurate and complete based ...
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,LLaMA Cleaned,mixed,The model correctly identified 'أذي نفسي' as a...,partial,The model captured the key entities: self-harm...,The model's output includes overlapping labels...
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,LLaMA Cleaned,mixed,The model correctly identified 'الوسواس' as a ...,partial,The model captured the core entities: the main...,The model's extraction of 'استغفري' as Treatme...
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,LLaMA Cleaned,accurate,The model correctly identified 'اكتئاب' and 'ف...,full,All relevant entities mentioned in the questio...,The model's extraction is accurate and complet...
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,LLaMA Cleaned,mixed,The model correctly identified the symptoms (ق...,partial,The model captured key entities such as sympto...,The model's extraction of 'الوسواس القهري' as ...


In [37]:
len(df_results)

499

In [38]:
files.download("NER_Evaluation_by_DeepSeek.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Allam

In [42]:
model_col = "ALLaM Cleaned"
judgment_field = "ALLaM Judgment"

# Keep the same results_dict if already defined
for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating {model_col}"):
    key = row["question"]

    # Initialize if new
    if key not in results_dict:
        results_dict[key] = {model_col: row[model_col]}
    else:
        results_dict[key][model_col] = row[model_col]

    # Skip if already evaluated
    if judgment_field in results_dict[key]:
        continue

    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)
    results_dict[key][judgment_field] = score

# Save
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)

Evaluating ALLaM Cleaned: 100%|██████████| 499/499 [18:25<00:00,  2.22s/it]


In [43]:
import json
from tqdm import tqdm

model_col = "ALLaM Cleaned"
judgment_field = "ALLaM Judgment"

# Load existing results
with open("ner_judgment_results.json", "r", encoding="utf-8") as f:
    results_dict = json.load(f)

for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Re-evaluating {model_col}"):
    key = row["question"]

    # Initialize model output in results if missing
    if key not in results_dict:
        results_dict[key] = {}
    results_dict[key][model_col] = row[model_col]

    # Always re-run ALLaM's judgment
    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)
    results_dict[key][judgment_field] = score

# Save updated JSON
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)

Re-evaluating ALLaM Cleaned: 100%|██████████| 499/499 [18:32<00:00,  2.23s/it]


In [44]:
from google.colab import files
files.download("ner_judgment_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [45]:
allam = pd.DataFrame(results_dict.values())

In [46]:
allam.insert(0, 'Question', df['question'])

In [47]:
allam.tail()

,Question,LLaMA Cleaned,LLaMA Judgment,GPT-4o Cleaned,GPT-4o Judgment,ALLaM Cleaned,ALLaM Judgment
494,قلق وتوتر ووسواس قهري وتأتأة في الكلام وعجز دا...,"[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""قلق، توتر، وسواس قهري، تأتأة في ...","{'correctness': {'verdict': 'mixed', 'reason':..."
495,اعاني من الخوف وتنبؤات من كلام اهلي وبصمتهم ان...,"[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""الخوف"", ""Diagnosis"": """", ""Treatm...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""الرهاب""، ""Diagnosis"": ""اضطراب ال...","{'correctness': {'verdict': 'poor', 'reason': ..."
496,لدي خمس سنوات علاج من الذهانية ولم اشفى تماما ...,"[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": """", ""Diagnosis"": ""الذهانية"", ""Tre...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""الذهانية"", ""Diagnosis"": ""الذهان""...","{'correctness': {'verdict': 'mixed', 'reason':..."
497,أحس بفقدان التركيز كأنني أنسى معلومة أخبرتني ب...,"[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": """", ""Diagnosis"": """", ""Treatment"":...","{'correctness': {'verdict': 'poor', 'reason': ...","[{""Symptom"": ""فقدان التركيز، النوم، الضغط، الغ...","{'correctness': {'verdict': 'poor', 'reason': ..."
498,انا احس فيني الاكتئاب الجزئي او عسر المزاج نفس...,"[{""Symptom"": ""الاكتئاب الجزئي، عسر المزاج، ماع...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""ماعندي شهيه، ماعندي شغف، مالي نف...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""الإكتئاب الجزئي، اعراض اكتئاب جز...","{'correctness': {'verdict': 'mixed', 'reason':..."


In [48]:
allam.to_excel("final_judgments.xlsx", index=False)

from google.colab import files
files.download("final_judgments.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [49]:
allam.to_csv("allam_judgments.csv", index=False)
files.download("allam_judgments.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## GPT-4o

In [39]:
model_col = "GPT-4o Cleaned"
judgment_field = "GPT-4o Judgment"

for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Evaluating {model_col}"):
    key = row["question"]

    # Initialize if new
    if key not in results_dict:
        results_dict[key] = {model_col: row[model_col]}
    else:
        results_dict[key][model_col] = row[model_col]

    # Skip if already evaluated
    if judgment_field in results_dict[key]:
        continue

    prompt = build_prompt(row["question"], row[model_col])
    score = score_with_deepseek(prompt)
    results_dict[key][judgment_field] = score

# Save
with open("ner_judgment_results.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)

Evaluating GPT-4o Cleaned: 100%|██████████| 499/499 [16:38<00:00,  2.00s/it]


In [50]:
gpt = pd.DataFrame(results_dict.values())

In [51]:
gpt.insert(0, 'Question', df['question'])

In [52]:
gpt.head()

,Question,LLaMA Cleaned,LLaMA Judgment,GPT-4o Cleaned,GPT-4o Judgment,ALLaM Cleaned,ALLaM Judgment
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,"[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","{'correctness': {'verdict': 'accurate', 'reaso...","[{""Symptom"": ""الخوف من عدم الإنجاب"", ""Diagnosi...","{'correctness': {'verdict': 'accurate', 'reaso...","[{""Symptom"": ""الخوف من عدم الإنجاب مستقبلاً"", ...","{'correctness': {'verdict': 'poor', 'reason': ..."
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,"[{""Symptom"": ""أذي نفسي"", ""Diagnosis"": """", ""Tre...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""أذي نفسي"", ""Diagnosis"": """", ""Tre...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""أذي نفسي"", ""Diagnosis"": """", ""Tre...","{'correctness': {'verdict': 'mixed', 'reason':..."
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,"[{""Symptom"": ""تكرار كلمة معينة، بلاحباط"", ""Dia...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""تكرار كلمة معينة، الاحباط"", ""Dia...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""تكرار كلمة معينة لمدة طويلة، الا...","{'correctness': {'verdict': 'poor', 'reason': ..."
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,"[{""Symptom"": ""انعزال، وانطوائية، وتعامل بسلبية...","{'correctness': {'verdict': 'accurate', 'reaso...","[{""Symptom"": ""انعزال، انطوائية، تعامل بسلبية"",...","{'correctness': {'verdict': 'accurate', 'reaso...","[{""Symptom"": ""اكتئاب، فوبيا من المجتمع, انعزال...","{'correctness': {'verdict': 'mixed', 'reason':..."
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,"[{""Symptom"": ""قلق شديد مزمن، خوف كبير، الوسواس...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""قلق شديد مزمن، خوف كبير"", ""Diagn...","{'correctness': {'verdict': 'mixed', 'reason':...","[{""Symptom"": ""قلق شديد، خوف، الوسواس القهري"", ...","{'correctness': {'verdict': 'mixed', 'reason':..."


In [53]:
gpt.to_excel("gpt_judgments.xlsx", index=False)

from google.colab import files
files.download("gpt_judgments.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [54]:
gpt.to_csv("gpt_judgments.csv", index=False)
files.download("gpt_judgments.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [55]:
files.download("ner_judgment_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [56]:
import pandas as pd


# Loop through each question entry in results_dict
for question, data in results_dict.items():
    for model_col in ["LLaMA Cleaned", "ALLaM Cleaned", "GPT-4o Cleaned"]:
        # Find the corresponding judgment field for that model
        judgment_field = model_col.replace("Cleaned", "Judgment").strip()

        if judgment_field in data:
            score = data[judgment_field]
            if isinstance(score, dict):
                flattened.append({
                    "Question": question,
                    "Model": model_col,
                    "Correctness": score["correctness"]["verdict"],
                    "Correctness Reason": score["correctness"]["reason"],
                    "Completeness": score["completeness"]["verdict"],
                    "Completeness Reason": score["completeness"]["reason"],
                    "Notes": score.get("notes", "")
                })

# Create a DataFrame and export
df_results = pd.DataFrame(flattened)
df_results.to_excel("NER_Evaluation_by_DeepSeek.xlsx", index=False)
print(f"Exported {len(df_results)} rows to NER_Evaluation_by_DeepSeek.xlsx")

Exported 1996 rows to NER_Evaluation_by_DeepSeek.xlsx


In [57]:
df_results.head()

,Question,Model,Correctness,Correctness Reason,Completeness,Completeness Reason,Notes
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,LLaMA Cleaned,accurate,The model correctly identified 'الخوف من عدم ا...,full,The only relevant entity in the question is th...,The extraction is accurate and complete based ...
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,LLaMA Cleaned,mixed,The model correctly identified 'أذي نفسي' as a...,partial,The model captured the key entities: self-harm...,The model's output includes overlapping labels...
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,LLaMA Cleaned,mixed,The model correctly identified 'الوسواس' as a ...,partial,The model captured the core entities: the main...,The model's extraction of 'استغفري' as Treatme...
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,LLaMA Cleaned,accurate,The model correctly identified 'اكتئاب' and 'ف...,full,All relevant entities mentioned in the questio...,The model's extraction is accurate and complet...
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,LLaMA Cleaned,mixed,The model correctly identified the symptoms (ق...,partial,The model captured key entities such as sympto...,The model's extraction of 'الوسواس القهري' as ...


In [58]:
df_results['Model'].unique()

array(['LLaMA Cleaned', 'ALLaM Cleaned', 'GPT-4o Cleaned'], dtype=object)

In [59]:
files.download("NER_Evaluation_by_DeepSeek.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [60]:
import json
with open("fewshot_judgments.json", "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=2)

from google.colab import files
files.download("fewshot_judgments.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [61]:
import pandas as pd
from collections import Counter

# خريطة الدرجات: accurate/full = 2, mixed/partial = 1, poor/low(missing/inaccurate) = 0
correctness_scale = {"accurate": 2, "mixed": 1, "poor": 0, "inaccurate": 0}
completeness_scale = {"full": 2, "partial": 1, "low": 0, "missing": 0}

model_fields = {"GPT-4o": "GPT-4o Judgment", "LLaMA": "LLaMA Judgment", "ALLaM": "ALLaM Judgment"}

rows = []
for model_name, field in model_fields.items():
    c_counts = Counter()
    p_counts = Counter()
    c_scores, p_scores = [], []
    for key, data in results_dict.items():
        judgment = data.get(field)
        if not isinstance(judgment, dict):
            continue
        c_verdict = judgment.get("correctness", {}).get("verdict", "error")
        p_verdict = judgment.get("completeness", {}).get("verdict", "error")
        c_counts[c_verdict] += 1
        p_counts[p_verdict] += 1
        if c_verdict in correctness_scale:
            c_scores.append(correctness_scale[c_verdict])
        if p_verdict in completeness_scale:
            p_scores.append(completeness_scale[p_verdict])

    rows.append({
        "Model": model_name,
        "Correctness Avg": round(sum(c_scores)/len(c_scores), 3) if c_scores else None,
        "Completeness Avg": round(sum(p_scores)/len(p_scores), 3) if p_scores else None,
        "Correctness Counts": dict(c_counts),
        "Completeness Counts": dict(p_counts),
        "N errors": c_counts.get("error", 0),
    })

summary_df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', None)
display(summary_df)

,Model,Correctness Avg,Completeness Avg,Correctness Counts,Completeness Counts,N errors
0,GPT-4o,1.191,1.386,"{'accurate': 105, 'mixed': 382, 'poor': 10, 'error': 2}","{'full': 208, 'partial': 273, 'low': 14, 'error': 2, 'missing': 2}",2
1,LLaMA,1.157,1.329,"{'accurate': 90, 'mixed': 396, 'inaccurate': 3, 'poor': 9, 'error': 1}","{'full': 182, 'partial': 298, 'low': 11, 'missing': 7, 'error': 1}",1
2,ALLaM,0.690,1.099,"{'poor': 165, 'mixed': 315, 'accurate': 13, 'error': 5, 'inaccurate': 1}","{'partial': 357, 'full': 93, 'low': 39, 'error': 5, 'missing': 5}",5
